# task_08 Solution

In [1]:

from pathlib import Path
import pandas as pd

base = Path("../../tasks/task_08/data/")


In [2]:

students = pd.read_csv(base / "students.csv")
courses = pd.read_csv(base / "courses.csv")
enrollments = pd.read_csv(base / "enrollments.csv")
attendance = pd.read_csv(base / "attendance.csv")
activities = pd.read_csv(base / "activities.csv")

Q1: Among students who took at least one STEM course and one Humanities course and never had a monthly attendance rate below 85%, which homeroom has the highest average attendance-penalized credit-weighted exam score?

In [3]:
students = pd.read_csv(base / "students.csv")
attendance = pd.read_csv(base / "attendance.csv")
enrollments = pd.read_csv(base / "enrollments.csv")
courses = pd.read_csv(base / "courses.csv")

attendance["monthly_attendance_rate"] = (attendance["school_days"] - attendance["absent_days"]) / attendance["school_days"]
worst_month = attendance.groupby("student_id")["monthly_attendance_rate"].min().rename("worst_month_rate")
overall_attendance = attendance.assign(attended_days=attendance["school_days"] - attendance["absent_days"]).groupby("student_id").apply(lambda g: g["attended_days"].sum() / g["school_days"].sum(), include_groups=False).rename("overall_attendance_rate")

enroll = enrollments.merge(courses, on="course_id")
weighted_exam = enroll.groupby("student_id").apply(lambda g: (g["exam_score"] * g["credit"]).sum() / g["credit"].sum(), include_groups=False).rename("weighted_exam")
subjects = enroll.groupby("student_id")["subject"].agg(lambda s: set(s))

student_scores = pd.concat([weighted_exam, overall_attendance, worst_month], axis=1).reset_index().merge(students, on="student_id")
student_scores["has_both_subject_groups"] = student_scores["student_id"].map(lambda sid: {"STEM", "Humanities"}.issubset(subjects[sid]))
eligible = student_scores[(student_scores["has_both_subject_groups"]) & (student_scores["worst_month_rate"] >= 0.85)].copy()
eligible["attendance_penalized_score"] = eligible["weighted_exam"] * eligible["overall_attendance_rate"]

q1 = str(eligible.groupby("homeroom")["attendance_penalized_score"].mean().sort_values(ascending=False).index[0])
print(q1)

HR5


Q4: Which student_id has the largest gap between their average homework score and average exam score across enrolled courses?

In [4]:
stud_diff = enrollments.groupby("student_id").agg(avg_homework=("homework_score", "mean"), avg_exam=("exam_score", "mean"))
stud_diff["diff"] = stud_diff["avg_homework"] - stud_diff["avg_exam"]
q4 = str(stud_diff.sort_values(["diff", "student_id"], ascending=[False, True]).index[0])
print(q4)

S07
